# Дом и квартира — эксперимент на материалах курса

Этот ноутбук запускает код из комплекта `home_course_materials.zip`. Новый эксперимент использует **50 train / 20 val / 30 bench**; старые 192 синтетических примера не подменяют данные курса.

В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**. Скачайте новый ZIP из сообщения с материалами и загрузите его в первой ячейке. Результаты сохраняются в отдельной папке Google Drive. Исходный полный ноутбук и весь код находятся внутри ZIP.

GPU-обучение и новые судейские оценки необходимо выполнить. Наличие этого ноутбука не означает, что эксперимент уже завершён.

## 1. Загрузка комплекта
Подключите Google Drive и выберите `home_course_materials.zip`. При повторном запуске уже распакованный проект используется без перезаписи.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, datetime
from google.colab import files, drive

USE_DRIVE = True
RUN_NAME = 'run_01'
if USE_DRIVE:
    drive.mount('/content/drive')
    RUN_FOLDER = Path('/content/drive/MyDrive/home_safe_sft_course') / RUN_NAME
else:
    RUN_FOLDER = Path('/content/home_safe_sft_course') / RUN_NAME
PROJECT = RUN_FOLDER / 'course_v2'
if not (PROJECT / 'src/pipeline.py').is_file():
    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zips) != 1:
        raise ValueError('Выберите один архив home_course_materials.zip.')
    RUN_FOLDER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        anchors = [n for n in z.namelist() if n.endswith('course_v2/src/pipeline.py')]
        if len(anchors) != 1:
            raise ValueError('Нужен новый комплект с папкой course_v2.')
        prefix = anchors[0][:-len('src/pipeline.py')]
        for info in z.infolist():
            if not info.filename.startswith(prefix):
                continue
            relative = info.filename[len(prefix):]
            if not relative:
                continue
            target = (PROJECT / relative).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise ValueError('Недопустимый путь в архиве')
            if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Символические ссылки не допускаются')
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(info))
if not (PROJECT / 'data/raw/home-sft-public-train.jsonl').is_file():
    raise FileNotFoundError('Не найдены новые данные курса')
full_notebook = json.loads((PROJECT / 'notebooks/02_home_course_colab.ipynb').read_text(encoding='utf-8'))
COURSE_STAGES = []
for cell in full_notebook['cells']:
    if cell['cell_type'] == 'code':
        source = cell['source']
        COURSE_STAGES.append(source if isinstance(source, str) else ''.join(source))
if len(COURSE_STAGES) != 8:
    raise ValueError('Неожиданная версия полного ноутбука; откройте notebook из ZIP напрямую')
print('Проект:', PROJECT)
print(subprocess.check_output(['nvidia-smi'], text=True))

## 2. Окружение
Изолированный Python 3.12.9 и закреплённые зависимости. Команды сохраняют логи. Код ячеек загружен из полного ноутбука в комплекте.

In [ ]:
exec(COURSE_STAGES[1], globals())

## 3. Тесты и аудит
Проверяются исходные данные, формат ответов, метки и оценка по рубрикам. Тестовые искусственные ответы не являются результатами модели.

In [ ]:
exec(COURSE_STAGES[2], globals())

## 4. Бейзлайн, обучение, сравнение
Последовательно выполняются prepare, два пробных ответа, генерация на val и bench, двухшаговый smoke test, полный SFT от исходной базы, новая генерация, автоматические метрики и экспорт 60 заданий судье.

Не меняйте параметры между бейзлайном и дообученной моделью. Длинные входы обрабатываются одинаково, сокращения протоколируются. При ошибке прочитайте последний журнал; не выдавайте незавершённый запуск за результат.

In [ ]:
exec(COURSE_STAGES[3], globals())

## 5. Независимая оценка без API-ключа
Скачайте архив **blind_packets_only.zip** и передайте другой модели-судье. Он содержит пять пакетов по 12 ответов и точную JSONL-схему. Не передавайте судье `mapping_private.jsonl` или полный архив результатов. Старые оценки J0001–J0072 для этого эксперимента не подходят.

In [ ]:
exec(COURSE_STAGES[4], globals())

## 6. Импорт оценок
Выберите один или несколько JSONL с оценками по критериям. Укажите фактическое название модели-судьи из интерфейса. Импорт проверяет идентификаторы, контрольные суммы и все критерии. Повторная загрузка одинаковых оценок не создаёт дубликаты.

In [ ]:
exec(COURSE_STAGES[5], globals())

## 7. Финальный отчёт и архив
После получения всех 60 оценок проверка требует новые веса, полные парные предсказания и метрики. Только после успешной проверки создаётся `home_course_READY_submission.zip`; PDF и Markdown заполняются фактическими результатами.

In [ ]:
exec(COURSE_STAGES[6], globals())

## 8. Необязательно: публикация в GitHub
После готовности комплекта ячейка запрашивает подтверждение и токен с правом Contents: Read and write только для `home-safe-sft`. Токен вводится в скрытое поле, не передаётся в чат и не записывается в URL. Публикуется отдельная папка course_v2; старый эксперимент сохраняется. Перед публикацией проверьте право размещать материалы курса в открытом доступе.

In [ ]:
exec(COURSE_STAGES[7], globals())